<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/Copy_of_cnn_rnn_tutorial_Lecture_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Deep Learning Hands-On: CNN & RNN from Scratch


### What we will cover
1. How images are stored as numbers (B&W → Grayscale → Colour)
2. What CNN filters do
3. Building and training a CNN on image classification
4. Why CNN struggles with text
5. Word embeddings and RNNs for text classification

### How to use this notebook
- **Pre-filled cells** — read, run, observe the output
- **✏️ DIY cells** — you write the code (hints are provided!)
- Run cells **in order**, top to bottom

---


## 0. Setup

Run this cell first. It imports everything we need for the whole notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("✅ All imports successful!")
print(f"   PyTorch version : {torch.__version__}")
print(f"   GPU available   : {torch.cuda.is_available()}")


---
## Part 1 — Images and Convolutional Neural Networks

---

### 1.1 Black & White Images

A **black & white image** is just a 2D grid of 0s and 1s.  
`0` = black pixel, `1` = white pixel. That's all.

The grid below spells out a letter "F".  
Run the cell — it will print the numbers and then display the image.


In [ ]:
bw = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
], dtype=np.float32)

print("The image as a matrix of numbers:")
print(bw)

plt.figure(figsize=(3, 3))
plt.imshow(bw, cmap='gray', vmin=0, vmax=1)
plt.title("Black & White (8×8)")
plt.axis('off')
plt.show()


---
#### ✏️ DIY 1 — Draw your own letter or shape

Edit the array below to draw something — your initials, a smiley, a cross, anything.  
Remember: `1` = white, `0` = black.


In [ ]:
# ✏️ YOUR TURN
# Edit the 0s and 1s below to draw your own 8×8 image

my_image = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
], dtype=np.float32)

plt.figure(figsize=(3, 3))
plt.imshow(my_image, cmap='gray', vmin=0, vmax=1)
plt.title("My image")
plt.axis('off')
plt.show()


---
### 1.2 Grayscale Images

Grayscale adds **shades** between black and white.  
Each pixel is now an integer from **0 (black) to 255 (white)**.

This is how the MNIST handwritten digit dataset stores its images — 28×28 grayscale.

The cell below creates a random grayscale image and annotates the pixel values.


In [ ]:
np.random.seed(7)
gray = np.random.randint(0, 256, size=(28, 28), dtype=np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# Left: full image
axes[0].imshow(gray, cmap='gray')
axes[0].set_title("Full 28×28 grayscale image")
axes[0].axis('off')

# Right: zoom into the top-left 6×6 corner, show values
corner = gray[:6, :6]
axes[1].imshow(corner, cmap='gray', vmin=0, vmax=255)
for r in range(6):
    for c in range(6):
        axes[1].text(c, r, str(corner[r, c]), ha='center', va='center',
                     fontsize=8, color='red', fontweight='bold')
axes[1].set_title("Top-left 6×6 corner (red = pixel value)")
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f"Image shape: {gray.shape}  |  min: {gray.min()}  max: {gray.max()}")


---
#### ✏️ DIY 2 — Solid colour images

What does a completely black image look like? What about a mid-grey one?  
Fill in the blanks below.


In [ ]:
# ✏️ YOUR TURN
# Hint: np.zeros((28, 28)) gives all zeros (black)
#       np.ones((28, 28))  gives all ones
#       multiply by a number to get a specific shade, e.g. np.ones(...) * 128

black_image = # write the code here

grey_image  = # write the code here

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(black_image, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Black")
axes[0].axis('off')
axes[1].imshow(grey_image, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("Grey")
axes[1].axis('off')
plt.show()


---
### 1.3 Colour (RGB) Images

A colour image has **3 channels stacked together**: Red, Green, Blue.  
Each channel is a grayscale grid. The full image has shape **(Height, Width, 3)**.

PyTorch uses a different convention: **(3, Height, Width)** — channels come first.


In [ ]:
np.random.seed(0)
rgb = np.random.randint(0, 256, size=(32, 32, 3), dtype=np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(rgb);                              axes[0].set_title("Full colour")
axes[1].imshow(rgb[:,:,0], cmap='Reds');          axes[1].set_title("Red channel")
axes[2].imshow(rgb[:,:,1], cmap='Greens');        axes[2].set_title("Green channel")
axes[3].imshow(rgb[:,:,2], cmap='Blues');         axes[3].set_title("Blue channel")
for ax in axes: ax.axis('off')
plt.suptitle("One image = three grayscale grids", y=1.02)
plt.tight_layout()
plt.show()

print(f"NumPy shape (H, W, C): {rgb.shape}")


In [ ]:
# Converting to a PyTorch tensor
# Step 1: permute — move channels from last to first
# Step 2: normalise — divide by 255 so values are 0.0 to 1.0

tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0

print(f"NumPy  shape  (H, W, C) : {rgb.shape}")
print(f"Tensor shape  (C, H, W) : {tuple(tensor.shape)}")
print(f"Value range             : {tensor.min():.2f}  to  {tensor.max():.2f}")


---
#### ✏️ DIY 3 — Make a solid red image

Create a 32×32 RGB image where every pixel is pure red (R=255, G=0, B=0)  
and display it with `imshow`.


In [ ]:
# ✏️ YOUR TURN
# Hint: start with np.zeros((32, 32, 3), dtype=np.uint8)
#       then set the Red channel: img[:, :, 0] = 255

red_image = # write the code here

plt.figure(figsize=(3, 3))
plt.imshow(red_image)
plt.title("Solid Red Image")
plt.axis('off')
plt.show()


---
### 1.4 What Do CNN Filters Do?

A **filter** (or kernel) is a small matrix — typically 3×3.  
It slides across the image, multiplies element-wise with each patch, and sums the result.  
This gives one number per patch — the **filter's response** at that location.

Different filters detect different patterns. Let's see three.


In [ ]:
# Test image — a bright square with a hollow centre
img = np.zeros((64, 64), dtype=np.float32)
img[8:56, 8:56] = 1.0     # outer square
img[22:42, 22:42] = 0.0   # hollow

def apply_filter(image, kernel):
    t = torch.from_numpy(image).unsqueeze(0).unsqueeze(0)   # (1, 1, H, W)
    k = torch.from_numpy(kernel.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        return F.conv2d(t, k, padding=1).squeeze().numpy()

f_horiz  = np.array([[-1.,-2.,-1.],[ 0., 0., 0.],[ 1., 2., 1.]])   # horizontal edges
f_vert   = np.array([[-1., 0., 1.],[-2., 0., 2.],[-1., 0., 1.]])   # vertical edges
f_blur   = np.ones((3, 3)) / 9.0                                     # average blur

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (title, data) in zip(axes, [
    ("Original",          img),
    ("Horizontal edges",  apply_filter(img, f_horiz)),
    ("Vertical edges",    apply_filter(img, f_vert)),
    ("Blur",              apply_filter(img, f_blur)),
]):
    ax.imshow(data, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle("Same image — four different filters", y=1.02)
plt.tight_layout()
plt.show()


---
#### ✏️ DIY 4 — Sharpening filter

A sharpening filter emphasises edges and makes the image look crisper.  
Fill in the kernel values and apply it to the same test image.

```
Sharpening kernel:
 [ 0, -1,  0]
 [-1,  5, -1]
 [ 0, -1,  0]
```


In [ ]:
# ✏️ YOUR TURN
# Define the sharpening filter using the values above

f_sharp = np.array([
    # write the code here (3 rows, 3 values each)
])

result = apply_filter(img, f_sharp)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img, cmap='gray');    axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(result, cmap='gray'); axes[1].set_title("Sharpened"); axes[1].axis('off')
plt.tight_layout()
plt.show()


---
### 1.5 From Image to Vector

A CNN stacks Conv + Pool layers to gradually compress the image.  
At the end, a **Flatten** turns the 3D feature maps into a 1D vector — the image's "summary".  
That vector goes into a fully-connected layer to make a class prediction.

```
(3, 32, 32) → Conv → Pool → Conv → Pool → Flatten → (vector) → FC → class
```

Run the cell to trace the shapes step by step.


In [ ]:
x = torch.randn(1, 3, 32, 32)
print(f"Input              : {tuple(x.shape)}")

c1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
x  = F.relu(c1(x))
print(f"After Conv1 + ReLU : {tuple(x.shape)}   ← 16 filters, spatial size unchanged (padding=1)")

x  = F.max_pool2d(x, 2)
print(f"After MaxPool      : {tuple(x.shape)}   ← halved by pooling")

c2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
x  = F.relu(c2(x))
print(f"After Conv2 + ReLU : {tuple(x.shape)}   ← 32 filters")

x  = F.max_pool2d(x, 2)
print(f"After MaxPool      : {tuple(x.shape)}   ← halved again")

x  = x.view(x.size(0), -1)
print(f"After Flatten      : {tuple(x.shape)}   ← this is the feature vector")
print(f"32×32 colour image  →  {x.shape[1]}-number summary")


---
#### ✏️ DIY 5 — What if the input is bigger?

Change the input to a **64×64** image and re-run the shape trace.  
What is the size of the final feature vector?


In [ ]:
# ✏️ YOUR TURN
# Change the input size from (1, 3, 32, 32) to (1, 3, 64, 64) and run

x = torch.randn(# write the code here)
print(f"Input              : {tuple(x.shape)}")

c1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
x  = F.relu(c1(x))
print(f"After Conv1 + ReLU : {tuple(x.shape)}")

x  = F.max_pool2d(x, 2)
print(f"After MaxPool      : {tuple(x.shape)}")

c2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
x  = F.relu(c2(x))
print(f"After Conv2 + ReLU : {tuple(x.shape)}")

x  = F.max_pool2d(x, 2)
print(f"After MaxPool      : {tuple(x.shape)}")

x  = x.view(x.size(0), -1)
print(f"After Flatten      : {tuple(x.shape)}")
print(f"Final feature vector size: {x.shape[1]}")


---
### 1.6 Building a CNN for CIFAR-10

**CIFAR-10** — 60,000 colour images (32×32), 10 classes:  
✈️ airplane · 🚗 car · 🐦 bird · 🐱 cat · 🦌 deer · 🐶 dog · 🐸 frog · 🐴 horse · 🚢 ship · 🚚 truck

Here is our model. Read through it — every line has a comment.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # --- Feature extractor ---
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1)  # 32×32 → 32×32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 16×16 → 16×16
        self.pool  = nn.MaxPool2d(2, 2)                            # halves H and W

        # --- Classifier ---
        # After 2 pools: 32→16→8, with 64 channels → 64*8*8 = 4096
        self.fc1  = nn.Linear(64 * 8 * 8, 256)
        self.fc2  = nn.Linear(256, 10)         # 10 output classes
        self.drop = nn.Dropout(0.4)            # randomly zero 40% during training

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # Conv → ReLU → Pool
        x = self.pool(F.relu(self.conv2(x)))   # Conv → ReLU → Pool
        x = x.view(x.size(0), -1)             # flatten to vector
        x = self.drop(F.relu(self.fc1(x)))
        x = self.fc2(x)                        # raw scores (no softmax — loss handles it)
        return x

model = SimpleCNN()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model ready. Total parameters: {total_params:,}")


In [ ]:
# Load CIFAR-10 data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std =[0.2470, 0.2435, 0.2616])
])

train_data   = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
test_data    = datasets.CIFAR10('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False)

print(f"Train: {len(train_data):,}  |  Test: {len(test_data):,}")


In [ ]:
# Peek at a batch of images before training
class_names = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
imgs, lbls = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.247, 0.243, 0.261]) + np.array([0.491, 0.482, 0.446])
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(class_names[lbls[i]], fontsize=8)
    ax.axis('off')
plt.suptitle("Sample training images", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Training loop
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_acc_hist, test_acc_hist = [], []
NUM_EPOCHS = 5

for epoch in range(1, NUM_EPOCHS + 1):
    # --- train ---
    model.train()
    correct = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == lbls).sum().item()
    train_acc_hist.append(correct / len(train_data))

    # --- evaluate ---
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(1) == lbls).sum().item()
    test_acc_hist.append(correct / len(test_data))

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS}  |  "
          f"train: {train_acc_hist[-1]:.1%}  |  test: {test_acc_hist[-1]:.1%}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_acc_hist, label='Train', color='steelblue', marker='o', markersize=4)
plt.plot(test_acc_hist,  label='Test',  color='tomato',    marker='o', markersize=4)
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("CNN on CIFAR-10"); plt.ylim(0, 1)
plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"Final test accuracy: {test_acc_hist[-1]:.1%}")


In [ ]:
# Training loop
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_acc_hist, test_acc_hist = [], []
NUM_EPOCHS = #write the code here

for epoch in range(1, NUM_EPOCHS + 1):
    # --- train ---
    model.train()
    correct = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == lbls).sum().item()
    train_acc_hist.append(correct / len(train_data))

    # --- evaluate ---
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(1) == lbls).sum().item()
    test_acc_hist.append(correct / len(test_data))

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS}  |  "
          f"train: {train_acc_hist[-1]:.1%}  |  test: {test_acc_hist[-1]:.1%}")


---
#### ✏️ DIY 6 — More filters in the first conv layer

Our `conv1` uses **32 filters**. Try increasing it to **64**.  
You will need to update `conv2`'s `in_channels` to match, and recalculate `fc1`'s input size.

Fill in the model below and train it. Does more filters help?


In [ ]:
# ✏️ YOUR TURN — increase the number of filters
# conv1: 3 → 64 filters (was 32)
# conv2: must take 64 as input (was 32)
# fc1  : input size changes — what is 64 * 8 * 8 ?  (stays the same actually — why?)

class WiderCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  # write the code here,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(# write the code here, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        self.drop  = nn.Dropout(0.4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

# Quick test — does the model run without errors?
test_model = WiderCNN()
dummy = torch.randn(2, 3, 32, 32)
print("Output shape:", test_model(dummy).shape)   # should be (2, 10)
print(f"Parameters: {sum(p.numel() for p in test_model.parameters()):,}")


In [ ]:
# training block goes here

---
#### ✏️ DIY 7 — Remove Dropout and observe overfitting

**Dropout** randomly zeroes out neurons during training — it acts as a regulariser  
and helps prevent **overfitting** (when the model memorises training data but fails on new data).

Copy the training loop from above, remove `self.drop` from the model, retrain, and compare the curves.  
You should see train accuracy climb much higher than test accuracy.


In [ ]:
# ✏️ YOUR TURN — train a model WITHOUT Dropout and compare curves

class NoDropCNN(nn.Module):   # fix the class name (remove the space)
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        # Dropout removed intentionally

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))   # no dropout here
        return self.fc2(x)

# write the training loop here (copy from the cell above, just swap the model)
# Then plot the curves and compare to the version WITH Dropout


In [ ]:
#Training block goes here

---
---
## Part 2 — Text and Recurrent Neural Networks

---

### 2.1 Why CNN Doesn't Work Well on Text

We just trained a great image classifier. The natural question is:  
*can we use the same CNN idea on text?*

Let's try. We will treat sentences as sequences of numbers and apply a 1D CNN.  
First, we need to turn words into numbers.


In [ ]:
# Our tiny dataset: 10 sentences, labelled positive (1) or negative (0)
texts = [
    "i love this movie it is fantastic",        # 1
    "great film wonderful acting",              # 1
    "absolutely amazing very enjoyable",        # 1
    "i enjoyed every moment of it",             # 1
    "this was a really good watch",             # 1
    "terrible movie i hated every minute",      # 0
    "awful boring complete waste of time",      # 0
    "i did not like this film at all",          # 0
    "worst movie ever very disappointing",      # 0
    "bad acting bad story bad everything",      # 0
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# Build a vocabulary — every unique word gets an index
all_words = []
for s in texts:
    all_words.extend(s.split())
vocab    = sorted(set(all_words))          # no <PAD> token needed for one-hot
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size : {VOCAB_SIZE} words")
print(f"  'love'     → index {word2idx['love']}")
print(f"  'terrible' → index {word2idx['terrible']}")


In [ ]:
# ── One-hot encoding ─────────────────────────────────────────────
# Instead of a single integer, each word becomes a vector of all zeros
# with a single 1 at its vocabulary index.
#
# Example (tiny vocab of 5 words):
#   "cat" at index 2  →  [0, 0, 1, 0, 0]
#   "dog" at index 4  →  [0, 0, 0, 0, 1]
#
# Advantage : each word is uniquely identified with no implied ordering.
# Disadvantage: the vectors are huge (size = vocab size) and carry
#               NO information about word meaning or similarity.

MAX_LEN = 12   # pad / truncate every sentence to this many words

def one_hot_encode(sentence):
    words = sentence.split()[:MAX_LEN]
    # Each word → a zero vector with a 1 at the word's index
    matrix = np.zeros((MAX_LEN, VOCAB_SIZE), dtype=np.float32)
    for t, word in enumerate(words):
        if word in word2idx:
            matrix[t, word2idx[word]] = 1.0
    # Rows beyond len(words) stay all-zero (padding)
    return matrix

# Encode all sentences → shape: (num_sentences, MAX_LEN, VOCAB_SIZE)
X_oh = np.stack([one_hot_encode(t) for t in texts])
X    = torch.from_numpy(X_oh)          # tensor shape: (10, 12, VOCAB_SIZE)
y    = torch.tensor(labels)

print(f"One-hot matrix shape per sentence : (MAX_LEN={MAX_LEN}, VOCAB_SIZE={VOCAB_SIZE})")
print(f"Full dataset tensor shape         : {tuple(X.shape)}")
print(f"\nFirst sentence  : '{texts[0]}'")
print(f"First word vector (one-hot of '{texts[0].split()[0]}'):")
print(X[0, 0].numpy())
print(f"  → 1.0 at position {int(X[0,0].argmax())}  ('{vocab[int(X[0,0].argmax())]}')")


In [ ]:
# 1D CNN on text — using one-hot vectors as input
# Each word is now a VOCAB_SIZE-dimensional vector (instead of a single integer).

class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Conv1d slides a 3-word window over the sequence
        # in_channels = VOCAB_SIZE  (each word = one-hot vector of that size)
        self.conv = nn.Conv1d(in_channels=VOCAB_SIZE, out_channels=32, kernel_size=3, padding=1)
        self.fc   = nn.Linear(32, 2)

    def forward(self, x):
        # x: (batch, MAX_LEN, VOCAB_SIZE)
        x = x.permute(0, 2, 1)                        # → (batch, VOCAB_SIZE, MAX_LEN)
        x = F.relu(self.conv(x)).max(dim=2).values     # → (batch, 32)
        return self.fc(x)

cnn_text = TextCNN()
opt_cnn  = torch.optim.Adam(cnn_text.parameters(), lr=1e-2)
crit     = nn.CrossEntropyLoss()

cnn_losses = []
for _ in range(300):
    cnn_text.train(); opt_cnn.zero_grad()
    loss = crit(cnn_text(X), y); loss.backward(); opt_cnn.step()
    cnn_losses.append(loss.item())

with torch.no_grad():
    acc = (cnn_text(X).argmax(1) == y).float().mean().item()
print(f"1D CNN (one-hot input) — training accuracy: {acc:.1%}")


The CNN memorised the training data. But it has a **fundamental weakness**:

> *"The food was **not** good."* — Negative  
> *"The food was **really** good."* — Positive

A 3-word filter window can catch `not good` — but what about this?

> *"Although everyone expected it to fail, the film was actually good."*

The word `fail` is 7 positions before `good`. The CNN's window **cannot** bridge that gap.

**The real problem:** text meaning often depends on words that are far apart.  
Negation, subject-verb agreement, coreference — all span long distances.

We need a model that reads words **one at a time** and keeps a **running memory**.  
That is the **Recurrent Neural Network**.


---
### 2.2 From One-Hot to Word Embeddings

One-hot vectors work, but they have two big problems:

| Problem | Example |
|---|---|
| **Huge size** | Vocab of 10,000 words → each vector has 10,000 numbers |
| **No meaning** | `love` and `like` are equally distant from each other as `love` and `hate` |

**Word embeddings** fix both problems.  
Each word is mapped to a small **dense vector** (e.g. 16 numbers) that is *learned during training*.  
Similar words end up with similar vectors.

```
One-hot  →  [0, 0, 0, 1, 0, 0, ...]   ← mostly zeros, no meaning
Embedding →  [0.42, -0.18, 0.91, ...]  ← compact, meaningful
```

This is the idea behind **Word2Vec** — train on a huge corpus until  
`vector("king") − vector("man") + vector("woman") ≈ vector("queen")`.

For now we build a simple learned embedding from scratch using `nn.Embedding`.  
There is a **TODO** at the end of this notebook to swap it for pre-trained Word2Vec!


In [ ]:
# Let's compare one-hot and learned embeddings side by side

EMBED_DIM = 16
emb = nn.Embedding(num_embeddings=VOCAB_SIZE, embedding_dim=EMBED_DIM)

# One word index
word  = "love"
idx   = word2idx[word]

# One-hot representation
oh = np.zeros(VOCAB_SIZE, dtype=np.float32)
oh[idx] = 1.0

# Learned embedding representation
vec = emb(torch.tensor(idx)).detach().numpy()

print(f"Word: '{word}'  (index {idx})")
print(f"\n--- One-hot ({VOCAB_SIZE}-dim) ---")
print(f"Non-zero count : 1 out of {VOCAB_SIZE}")
print(f"Vector snippet : {oh[:10]}  ...")
print(f"\n--- Learned embedding ({EMBED_DIM}-dim) ---")
print(f"Non-zero count : all {EMBED_DIM}")
print(f"Vector         : {vec.round(3)}")
print(f"\nKey point: the embedding is {VOCAB_SIZE//EMBED_DIM}x smaller and will become")
print(f"meaningful as the model trains (similar words → similar vectors).")


---
### 2.3 The RNN

An RNN processes words one at a time, updating a **hidden state** `h` at each step:

```
h_t  =  tanh( W_h · h_{t-1}  +  W_x · x_t )
              ↑ memory             ↑ current word
```

After the last word, the final `h_T` captures the whole sentence → classify it.

```
"i"       → RNN → h1
"love"    → RNN → h2
"this"    → RNN → h3
"movie"   → RNN → h4  →  Linear  →  Positive 😊
```


In [ ]:
# For the RNN we switch from one-hot to learned embeddings.
# That means the input is integer indices again (one per word),
# and nn.Embedding does the lookup inside the model.

MAX_LEN = 12

def encode_idx(sentence):
    """Encode sentence as integer indices, pad with VOCAB_SIZE (safe OOV index)."""    words = sentence.split()[:MAX_LEN]
    ids   = [word2idx.get(w, 0) for w in words]
    ids  += [0] * (MAX_LEN - len(ids))
    return ids

X_idx = torch.tensor([encode_idx(t) for t in texts])   # (10, 12)
# y is the same as before


class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # Embedding: integer index → dense vector (learned during training)
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.emb(x)                      # (batch, seq, embed_dim)
        _, hidden = self.rnn(x)             # hidden: (1, batch, hidden_dim)
        return self.fc(hidden.squeeze(0))   # (batch, 2)


rnn_model = SentimentRNN(VOCAB_SIZE, embed_dim=16, hidden_dim=32)
opt_rnn   = torch.optim.Adam(rnn_model.parameters(), lr=1e-2)

rnn_losses = []
for _ in range(300):
    rnn_model.train(); opt_rnn.zero_grad()
    loss = crit(rnn_model(X_idx), y); loss.backward(); opt_rnn.step()
    rnn_losses.append(loss.item())

with torch.no_grad():
    acc = (rnn_model(X_idx).argmax(1) == y).float().mean().item()
print(f"RNN (learned embeddings) — accuracy: {acc:.1%}")

# Compare all three loss curves
plt.figure(figsize=(8, 3))
plt.plot(cnn_losses, label='1D CNN (one-hot)',    color='tomato',    linewidth=2)
plt.plot(rnn_losses, label='RNN (learned emb.)',  color='steelblue', linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("1D CNN (one-hot input) vs RNN (learned embeddings)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Test the trained RNN on new sentences
def predict(sentence, model):
    model.eval()
    with torch.no_grad():
        out  = model(torch.tensor([encode_idx(sentence)]))
        prob = torch.softmax(out, dim=1)[0]
        pred = prob.argmax().item()
    return ("Positive 😊" if pred == 1 else "Negative 😞"), prob[pred].item()

print("RNN predictions on new sentences:\n")
for s in ["i really liked this movie",
          "this film was so boring",
          "wonderful and very enjoyable",
          "absolutely terrible experience"]:
    lbl, conf = predict(s, rnn_model)
    print(f"  '{s}'")
    print(f"   → {lbl}  ({conf:.0%})\n")


---
## ✏️ DIY Exercises — RNN Variations

Now it's your turn. Each exercise below is a focused, small change.  
Run them in order — each builds on the model and helpers defined above.

---

### Exercise A — Change the hidden size

`hidden_dim` controls how much memory the RNN has at each step.  
Try very small (4) and larger (128). On this tiny dataset bigger is not always better.


In [ ]:
# ✏️ YOUR TURN
# Change HIDDEN_DIM and re-run. Try: 4, 16, 64, 128

HIDDEN_DIM = 32   # ← change this

model_a = SentimentRNN(VOCAB_SIZE, embed_dim=16, hidden_dim=HIDDEN_DIM)
opt_a   = torch.optim.Adam(model_a.parameters(), lr=1e-2)

for _ in range(300):
    model_a.train(); opt_a.zero_grad()
    loss = crit(model_a(X_idx), y); loss.backward(); opt_a.step()

with torch.no_grad():
    acc = (model_a(X_idx).argmax(1) == y).float().mean().item()
print(f"hidden_dim = {HIDDEN_DIM}  →  accuracy: {acc:.1%}")


---
### Exercise B — Add more training sentences

The model currently trains on only 10 sentences.  
Add at least 4 more (2 positive, 2 negative), retrain, and see if anything changes.


In [ ]:
# ✏️ YOUR TURN
# Add your own sentences and matching labels below

extra_texts = [
    "write a positive sentence here",    # 1
    "write another positive sentence",   # 1
    "write a negative sentence here",    # 0
    "write another negative sentence",   # 0
]
extra_labels = [1, 1, 0, 0]

all_texts  = texts  + extra_texts
all_labels = labels + extra_labels

X2 = torch.tensor([encode_idx(t) for t in all_texts])
y2 = torch.tensor(all_labels)

model_d = SentimentRNN(VOCAB_SIZE, embed_dim=16, hidden_dim=32)
opt_d   = torch.optim.Adam(model_d.parameters(), lr=1e-2)

for _ in range(300):
    model_d.train(); opt_d.zero_grad()
    loss = crit(model_d(X2), y2); loss.backward(); opt_d.step()

with torch.no_grad():
    acc = (model_d(X2).argmax(1) == y2).float().mean().item()
print(f"Dataset size: {len(all_texts)} sentences  →  accuracy: {acc:.1%}")


---
### Exercise C — Two-layer RNN

An RNN can be stacked: the hidden state of layer 1 becomes the input to layer 2.  
Add `num_layers=2` to `nn.RNN` and compare training convergence.


In [ ]:
# ✏️ YOUR TURN
# Add num_layers=2 to nn.RNN and extract the correct hidden state.

class TwoLayerRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim,
                          num_layers= # write the code here,
                          batch_first=True)
        self.fc  = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.emb(x)
        _, hidden = self.rnn(x)
        # hidden shape is (num_layers, batch, hidden_dim)
        # use hidden[-1] to get the last layer's hidden state
        return self.fc(hidden[-1])

model_e = TwoLayerRNN(VOCAB_SIZE, embed_dim=16, hidden_dim=32)
opt_e   = torch.optim.Adam(model_e.parameters(), lr=1e-2)

e_losses = []
for _ in range(300):
    model_e.train(); opt_e.zero_grad()
    loss = crit(model_e(X_idx), y); loss.backward(); opt_e.step()
    e_losses.append(loss.item())

with torch.no_grad():
    acc = (model_e(X_idx).argmax(1) == y).float().mean().item()
print(f"2-layer RNN  →  accuracy: {acc:.1%}")

plt.figure(figsize=(7, 3))
plt.plot(rnn_losses, label='1-layer RNN', color='steelblue', linewidth=2)
plt.plot(e_losses,   label='2-layer RNN', color='seagreen',  linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("1-layer vs 2-layer RNN")
plt.legend(); plt.grid(alpha=0.3)
plt.show()


---
### 🔧 TODO — Upgrade to Pre-trained Word2Vec Embeddings

So far our `nn.Embedding` starts with **random vectors** and learns from scratch.  
Real NLP pipelines load **pre-trained embeddings** (trained on billions of words) so the  
model already knows that "fantastic" and "wonderful" are similar before it sees a single training example.

**Your task:** replace the random embedding in `SentimentRNN` with pre-trained Word2Vec vectors.

Steps:
1. `pip install gensim`
2. Load the pre-trained model  (`word2vec-google-news-300`)
3. Build an embedding matrix for your vocabulary
4. Load it into `nn.Embedding` and (optionally) freeze it

The template cell below has the skeleton — fill in the blanks.


In [ ]:
# 🔧 TODO — Pre-trained Word2Vec embeddings
# Run: !pip install gensim   (once, in a Colab cell)

import gensim.downloader as api

# Step 1 — load pre-trained vectors (~1.6 GB download, 300-dim)
w2v = # Write your code here
EMBED_DIM_W2V = # Write your code here
print(f"Loaded! Example: vector for 'king' has shape {w2v['king'].shape}")

# Step 2 — build an embedding matrix for our vocabulary
embed_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM_W2V), dtype=np.float32)
found = 0
for word, idx in word2idx.items():
    if word in w2v:
        # # Write your code here
        found += 1
print(f"Found {found}/{VOCAB_SIZE} words in Word2Vec vocabulary")

# Step 3 — create an Embedding layer pre-loaded with those vectors
pretrained_emb = # Write your code here
pretrained_emb.weight.data = torch.tensor(embed_matrix)
pretrained_emb.weight.requires_grad = True   # True = fine-tune, False = freeze

# Step 4 — plug it into SentimentRNN
class SentimentRNN_W2V(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = # Write your code here                         # ← pre-trained!
        self.rnn = # Write your code here
        self.fc  = # Write your code here
    def forward(self, x):
        x = # Write your code here
        _, h = # Write your code here
        return self.fc(h.squeeze(0))

# Step 5 — train and compare accuracy with the random-embedding version

print("Compare final accuracy: random init vs Word2Vec pre-trained.")


---
## Wrap-up

| | CNN | RNN |
|---|---|---|
| Best input | Images, fixed-size grids | Sequences: text, time series |
| Local patterns | ✅ Great | ✅ Limited by window |
| Long-range memory | ❌ No | ✅ Yes |
| Word representation | One-hot / none | One-hot → Embedding → Word2Vec |

**Word representation ladder (worst → best):**  
`one-hot` → `random nn.Embedding` → `pre-trained Word2Vec` → `contextual (BERT)`

**What comes next?**  
- **LSTM** — adds a "cell state" to the RNN, much better at remembering over long sequences  
- **Transformer** — processes the whole sequence at once; powers BERT, GPT, and most modern NLP
